In [1]:
import time
import warnings
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import display
import ipywidgets as widgets
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import lightkurve as lk
from lightkurve import search_lightcurve
import warnings
from astropy.units import UnitsWarning


/Users/wayfinder/Code/fault-in-our-stars/.env/lib/python3.13/site-packages/lightkurve/prf/__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


In [2]:
import pandas as pd
from tqdm.auto import tqdm
import lightkurve as lk

def fetchStitchedLC(KIC):
    """
    Given a KIC, fetch its light curve data using Lightkurve.
    Returns: stitched LC object.
    """
    search = lk.search_lightcurve(
        f"KIC {KIC}",
        mission="Kepler",
        author="Kepler",
        cadence="long"
    )
    return search.download_all().stitch().remove_nans()

def getLCArrays(kic):
    lc = fetchStitchedLC(kic)

    return pd.Series({
        "time": lc.time.value,
        "flux": lc.flux.value,
        "flux_err": lc.flux_err.value if lc.flux_err is not None else None,
        "n_points": len(lc),
    })

def getLCArrays_safe(kic):
    try:
        return getLCArrays(kic)
    except Exception:
        return pd.Series({
            "time": None,
            "flux": None,
            "flux_err": None,
            "n_points": 0,
        })

def add_lightcurve_features_in_batches(df, batch_size=100):
    """
    df must be indexed by KIC.
    Processes rows in batches and appends light-curve features.
    """
    kics = df.index.to_list()
    out = []

    for start in tqdm(range(0, len(kics), batch_size), desc="Processing batches"):
        batch_kics = kics[start:start + batch_size]
        batch_features = [getLCArrays_safe(kic) for kic in tqdm(batch_kics, leave=False, desc="KICs in batch")]
        out.extend(batch_features)

    feature_df = pd.DataFrame(out, index=df.index)
    return df.join(feature_df)



In [7]:
df = pd.read_csv(r"../assets/data/kepler-eclipsing-binary-catalog.csv")
df = df.drop(columns=["Unnamed: 11"])
# df = df.head(150)
df = df.set_index("KIC")


In [ ]:
df


,period,period_err,bjd0,bjd0_err,morph,GLon,GLat,kmag,Teff,SC
KIC,,,,,,,,,,
3863594,0.053268,0.0,55000.000000,0.004327,0.79,-1.0000,-1.0000,-1.000,-1.0,False
10417986,0.073731,0.0,55000.027476,0.004231,0.99,81.0390,11.0820,9.128,-1.0,True
8912468,0.094838,0.0,54953.576945,0.005326,0.98,80.1095,7.8882,11.751,6194.0,False
8758716,0.107205,0.0,54953.672989,0.006197,1.00,77.7478,11.6565,13.531,-1.0,False
10855535,0.112782,0.0,54964.629315,0.006374,0.99,79.3949,15.9212,13.870,7555.0,False
...,...,...,...,...,...,...,...,...,...,...
9408440,989.985000,-1.0,55346.365980,0.096130,0.00,78.5607,12.2615,13.199,5688.0,False
8054233,1058.000000,-1.0,54751.806288,0.968052,0.03,78.6142,7.7321,11.783,4733.0,False
7672940,1064.270000,-1.0,54977.092960,0.089646,0.00,74.5296,14.6136,12.328,-1.0,False


In [ ]:
# Example:
df_with_lc = add_lightcurve_features_in_batches(df, batch_size=50)


Processing batches:   0%|          | 0/59 [00:00<?, ?it/s]

KICs in batch:   0%|          | 0/50 [00:00<?, ?it/s]

KICs in batch:   0%|          | 0/50 [00:00<?, ?it/s]

KICs in batch:   0%|          | 0/50 [00:00<?, ?it/s]

KICs in batch:   0%|          | 0/50 [00:00<?, ?it/s]

...

KICs in batch:   0%|          | 0/50 [00:00<?, ?it/s]

KICs in batch:   0%|          | 0/50 [00:00<?, ?it/s]

KICs in batch:   0%|          | 0/50 [00:00<?, ?it/s]

KICs in batch:   0%|          | 0/50 [00:00<?, ?it/s]

KICs in batch:   0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
# If KIC is the index:
# df_with_lc = download_lightcurves(df, max_workers=8)

# If KIC is in a column:
# df_with_lc = parallel_fetch_lightcurves(df, max_workers=8, kic_col="KIC")


In [ ]:
df_with_lc


,period,period_err,bjd0,bjd0_err,morph,GLon,GLat,kmag,Teff,SC,time,flux,flux_err,n_points
KIC,,,,,,,,,,,,,,
3863594,0.053268,0.000000e+00,55000.000000,0.004327,0.79,-1.0000,-1.0000,-1.000,-1.0,False,"[120.53895485026442, 120.55938947320828, 120.5...","[1.0027486, 0.9967121, 0.9994276, 0.9998087, 0...","[9.646405e-05, 9.793051e-05, 9.82528e-05, 9.78...",52286
10417986,0.073731,0.000000e+00,55000.027476,0.004231,0.99,81.0390,11.0820,9.128,-1.0,True,"[120.53855550513981, 120.55898988617264, 120.5...","[0.99979115, 0.9999734, 0.99954855, 0.9999474,...","[1.3734762e-05, 1.36594945e-05, 1.3738576e-05,...",52200
8912468,0.094838,0.000000e+00,54953.576945,0.005326,0.98,80.1095,7.8882,11.751,6194.0,False,"[120.53837293142715, 120.55880740127031, 120.5...","[0.9991674, 0.99967396, 0.9959102, 1.0009913, ...","[5.4617427e-05, 5.4451953e-05, 5.4505457e-05, ...",65266
8758716,0.107205,0.000000e+00,54953.672989,0.006197,1.00,77.7478,11.6565,13.531,-1.0,False,"[120.53883156277152, 120.55926600434032, 120.5...","[1.0111475, 0.9915452, 1.0019422, 1.0050972, 0...","[0.00011227019, 0.000111015965, 0.000111714115...",65262
10855535,0.112782,0.000000e+00,54964.629315,0.006374,0.99,79.3949,15.9212,13.870,7555.0,False,"[131.51221361904754, 131.5326480107833, 131.55...","[0.9962518, 1.004716, 1.0054883, 0.9961573, 1....","[0.0001593724, 0.0001598708, 0.0001599348, 0.0...",64793
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5699617,0.288917,2.000000e-07,54964.792095,0.016878,0.77,72.8265,12.4827,14.971,4817.0,False,"[131.51258611839148, 131.53302070768405, 131.5...","[0.79169065, 0.9118392, 1.0295155, 1.0966845, ...","[0.0002694164, 0.00028266877, 0.00029181127, 0...",64797
6467389,0.288976,2.000000e-07,54953.686178,0.017779,0.79,76.6095,7.3213,13.132,5183.0,False,"[120.53860112484108, 120.55903568393114, 120.5...","[0.8615235, 0.7217549, 0.8804035, 0.9540212, 1...","[9.8657554e-05, 0.000101613055, 0.00010539877,...",65266
9882280,0.289075,2.000000e-07,54964.877130,0.015868,0.77,77.6393,16.3034,14.516,5343.0,False,"[131.51238090716652, 131.5328153181108, 131.55...","[1.1296804, 1.1862673, 1.0375203, 0.7888919, 0...","[0.00022931161, 0.00022601815, 0.00021656218, ...",64791


In [ ]:
df_with_lc.to_json("../assets/data/keb-lcs.json")
